![Week 1 Day 3](../../assets/banners/w1d3.png)

# Week 1 Day 3 - Solution: Extracting Question 5

The Day 3 exercise: extract **Question 5** (the paging calculation) with its five mark points, add a `reference_topics` field, add a second check to `problems()`, and run the self-correcting loop, rehearsing first with the pretend model.

> 💡 Try the exercise yourself first (it's at the end of `week1/day3.ipynb`). Struggling a little is where the learning happens! Then come back and compare.

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import os
from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field

from gradewise import datasets
from gradewise.config import DATA_DIR, settings
from gradewise.exam.parse import parse_pdf
from gradewise.models import chat_model

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
        print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(settings.describe_models())
print(datasets.describe())

## Part 1: The Day 3 building blocks

Copied from `day3.ipynb` so this notebook runs on its own.

In [ ]:
exam_dir = DATA_DIR / "exams" / "cs_midterm"
paper_text = parse_pdf(exam_dir / "question_paper.pdf").text
scheme_text = parse_pdf(exam_dir / "marking_scheme.pdf").text


def between(text: str, start: str, end: str) -> str:
    i = text.index(start)
    return text[i : text.index(end, i)].strip()


class ExtractedQuestion(BaseModel):
    """One question from an exam paper."""

    number: str = Field(description='The question number as printed, e.g. "3" or "4(b)"')
    text: str = Field(description="The full question text, exactly as printed, without the marks")
    max_marks: float = Field(gt=0, description="The marks shown in square brackets")
    question_type: Literal["definition", "short", "long", "compare", "numerical"]


class Criterion(BaseModel):
    """One mark-earning point from the marking scheme."""

    description: str = Field(description="What the student must say or do, in the scheme's words")
    marks: float = Field(gt=0)


class QuestionWithCriteria(ExtractedQuestion):
    criteria: list[Criterion] = Field(min_length=1, description="Every mark point in the scheme, in order")

    def problems(self) -> list[str]:
        total = sum(c.marks for c in self.criteria)
        if abs(total - self.max_marks) > 0.01:
            return [f"The criteria add up to {total:g} marks, but the question is worth {self.max_marks:g}."]
        return []


class PretendModel:
    def __init__(self, answers: list[dict]):
        self.answers = list(answers)

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(lambda messages: schema.model_validate(self.answers.pop(0)))

## Part 2: Cut out Question 5

💡 Question 5 straddles the page break. An early version of `parse_pdf` let the page-1 footer ("Page 1 of 2 (turn over)") leak in here; the parser now strips known running lines even when they appear once (`KNOWN_RUNNING_LINES` in `gradewise/exam/parse.py`, see Day 5). Check the printout: no footer.

In [ ]:
q5_paper = between(paper_text, "**Question 5**", "## **Unit 2")
q5_scheme = between(scheme_text, "## **Question 5", "## **Question 6")
print(q5_paper, "\n" + "─" * 60)
print(q5_scheme)

## Part 3: The extended schema

Two changes: a new field (with a description written *for the model*), and a second check. Because `problems()` returns a list, adding a check is just appending to it, and `super()` keeps the original one.

In [ ]:
class Q5Extraction(QuestionWithCriteria):
    reference_topics: list[str] = Field(
        min_length=1,
        max_length=4,
        description="1-4 short CS201 course topics a grader should look up, e.g. ['paging', 'page tables']",
    )

    def problems(self) -> list[str]:
        issues = super().problems()  # 💡 keep the "criteria add up" check
        for i, c in enumerate(self.criteria, start=1):
            if len(c.description.strip()) < 10:
                issues.append(
                    f"Criterion {i} is too short to mark against: {c.description!r}. Copy the scheme's wording."
                )
        return issues


def extract_with_feedback(llm, messages: list, schema=Q5Extraction, max_attempts: int = 3):
    """Day 3's loop, with the schema as a parameter."""
    extractor = llm.with_structured_output(schema)
    history = list(messages)
    for attempt in range(1, max_attempts + 1):
        answer = extractor.invoke(history)
        issues = answer.problems()
        if not issues:
            print(f"✅ attempt {attempt}: clean")
            return answer
        print(f"⚠️ attempt {attempt}: {issues}")
        fix = "Your answer has these problems:\n" + "\n".join(issues) + "\nFix them and answer again."
        history += [AIMessage(answer.model_dump_json()), HumanMessage(fix)]
    print("🙋 still not clean: a human should check this one")
    return answer

## Part 4: Rehearse with the pretend model

First answer: one lazy criterion ("bits") and a missing mark point. Second answer: fixed.

In [ ]:
base = {
    "number": "5",
    "text": "A system has 32-bit logical addresses, a page size of 8 KB and page-table entries of 4 bytes. ...",
    "max_marks": 5,
    "question_type": "numerical",
    "reference_topics": ["paging", "page-table size calculation"],
}
lazy = base | {
    "criteria": [
        {"description": "bits", "marks": 1},
        {"description": "Number of pages = 2^19 = 524,288", "marks": 1},
        {"description": "Page table size = 2^19 × 4 bytes", "marks": 1},
        {"description": "Final answer 2 MB", "marks": 1},
    ]
}
good = base | {
    "criteria": [
        {"description": "Offset bits: 8 KB = 2^13 bytes, so 13 bits", "marks": 1},
        {"description": "Page-number bits = 32 - 13 = 19", "marks": 1},
        {"description": "Number of pages = 2^19 = 524,288", "marks": 1},
        {"description": "Method: page-table size = number of pages × entry size", "marks": 1},
        {"description": "Final answer: 2^21 bytes = 2 MB", "marks": 1},
    ]
}

rehearsal = extract_with_feedback(PretendModel([lazy, good]), ["(prompt)"])
print(rehearsal.reference_topics)

## Part 5: The real extraction

In [ ]:
messages = [
    (
        "system",
        "You are GradeWise's exam-understanding assistant. From the question and its marking scheme, "
        "extract the question and EVERY mark point as a criterion, in the scheme's words. Ignore page "
        "headers and footers. Never invent marks.",
    ),
    ("human", f"=== QUESTION ===\n{q5_paper}\n\n=== MARKING SCHEME ===\n{q5_scheme}"),
]

q5 = extract_with_feedback(chat_model(temperature=0), messages)
print("\nText:", q5.text)
print("Topics:", q5.reference_topics)
for c in q5.criteria:
    print(f"  [{c.marks:g}] {c.description}")

**Check the result against the answer key:** `expected_exam.json` has five 1-mark criteria for Q5 (`Q5.c1` to `Q5.c5`), and the text should be exactly the printed question. If the model paraphrases or drops the "Show your working" sentence, add a rule to the system prompt, the same way `gradewise/exam/extract.py`'s `SYSTEM_PROMPT` grew one rule per mistake.